# MySQL Data Factory - 批量插入工作流

## 工作流程
1. 连接数据库
2. 查询样本记录
3. 保存样本文件
4. 生成待插入数据
5. 执行批量插入

### 1️⃣ 导入模块

In [ ]:
import sys
sys.path.insert(0, '..')

from src.database import DatabaseManager
from src.data_generator import DataGenerator
from src.bulk_inserter import BulkInserter
from src.data_loader import DataLoader
import pandas as pd
from dotenv import load_dotenv

print("✓ 模块导入成功")

### 2️⃣ 连接数据库

In [ ]:
# 加载环境变量
load_dotenv()

# 连接数据库
db = DatabaseManager()
db.connect()

print("✓ 数据库连接成功")

### 3️⃣ 查询样本记录

In [ ]:
# 输入表名
table_name = "t_legal"  # 修改为你要操作的表

# 查询1条记录作为样本
sample_sql = f"SELECT * FROM {table_name} LIMIT 1"
sample_df = db.to_dataframe(sample_sql)

print(f"✓ 查询到样本记录：{len(sample_df)} 行")
print(f"\n样本数据预览：")
sample_df.head()

### 4️⃣ 保存样本文件

In [ ]:
# 保存样本到文件
loader = DataLoader()
sample_file = loader.save_sample(sample_df, table_name, file_type='csv')

print(f"✓ 样本已保存到：{sample_file}")
print(f"\n提示：你可以编辑 data/samples/ 目录下的文件，或参考样本生成新数据")

### 5️⃣ 生成待插入数据

In [ ]:
# 方式1：从样本生成（推荐）
generator = DataGenerator(locale='ja_JP')

# 生成数量
count = 100  # 修改为你需要的数量

# 从样本生成数据（主键会自动递增）
primary_key = 'LEGAL_ID'  # 修改为你的主键列名
generated_df = generator.generate_from_sample(sample_df, count=count, primary_key_col=primary_key)

print(f"✓ 生成了 {len(generated_df)} 条数据")
print(f"\n生成的数据预览：")
generated_df.head()

### 6️⃣ 保存待插入文件

In [ ]:
# 保存待插入数据
pending_file = loader.save_pending(generated_df, f"{table_name}_pending.csv")

print(f"✓ 待插入文件已保存：{pending_file}")
print(f"\n提示：你可以编辑 data/pending/ 目录下的文件，然后执行插入")

### 7️⃣ 执行批量插入

In [ ]:
# 创建批量插入器
inserter = BulkInserter(db)

# 执行批量插入
batch_size = 100  # 每批插入数量
inserted_count = inserter.insert_from_dataframe(
    generated_df, 
    table_name, 
    batch_size=batch_size
)

print(f"\n✓ 成功插入 {inserted_count} 条记录到 {table_name}")

### 8️⃣ 验证插入结果

In [ ]:
# 验证插入结果
count_sql = f"SELECT COUNT(*) FROM {table_name}"
total_count = db.query(count_sql)[0][0]

print(f"✓ 表 {table_name} 当前总记录数：{total_count}")

# 查看最新插入的5条记录
latest_sql = f"SELECT * FROM {table_name} ORDER BY UPDATED_AT DESC LIMIT 5"
latest_df = db.to_dataframe(latest_sql)

print(f"\n最新插入的记录：")
latest_df

### 9️⃣ 关闭连接

In [ ]:
# 关闭数据库连接
db.disconnect()

print("✓ 数据库连接已关闭")
print("\n🎉 工作流执行完成！")